# Explore pretrained C3PO models

**Run the two cells below once** (or choose **Runtime → Run all**). Then use the widgets to choose a recording and click **Load recording**. The trial explorer opens automatically.

Choose **Align to** and **Group by**, then click **Update plot**. The defaults are `start_time` and `reward` when available. Expand **Plot settings** to change the time window or bin a numeric feature.

Code is hidden by default in Colab. A CPU runtime is sufficient; large recordings may need more memory. No training or raw spike extraction is needed for the preview.

In [ ]:
# @title 1. Install C3PO
import subprocess
import sys

print("Installing C3PO and widget support…")
installation = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "c3po-neuro[cloud]",
        "ipywidgets",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
if installation.returncode:
    print(installation.stdout[-8000:])
    raise RuntimeError("Installation failed. See the output above, then retry.")
print("Ready. Run cell 2 to open the explorer.")

In [ ]:
# @title 2. Open the C3PO explorer
from __future__ import annotations

import gc
import html
import traceback
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display

from c3po.analysis.analysis import C3poAnalysis
from c3po.utils.dandi import get_dandi_file, get_spike_data

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

if globals().get("nwb_io") is not None:
    nwb_io.close()
for old_widget in globals().get("_explorer_widgets", []):
    old_widget.close()

HF_REPO = "samuelbray32/c3po-publication"
HF_REVISION = "main"
HF_CACHE_DIR = Path.cwd() / "c3po_cloud_cache"

# These variables remain available for additional analysis in new cells.
analysis = nwb_io = nwbfile = trials = selected_model = None
align_to = group_by = None
mark_times = mark_ids = aligned_result = None
_analysis_name = _loaded_name = None
_context_ready = False
_configuring = False
_busy = False


def scalar_columns(frame: pd.DataFrame) -> list[str]:
    """Return trial columns containing scalar values rather than arrays.

    Parameters
    ----------
    frame : pandas.DataFrame
        Trial metadata, with shape (n_trials, n_features).

    Returns
    -------
    list of str
        Scalar column names suitable for widget selections.
    """
    return [
        name for name in frame.columns if frame[name].map(pd.api.types.is_scalar).all()
    ]


def prepare_context() -> None:
    """Interpolate once and fit the loaded model's PCA on a regular time grid.

    Notes
    -----
    The 20 ms grid weights recording time uniformly and limits preview memory.
    All fitted PCs remain on ``analysis``; plots display the first three.
    Invalid context rows are normalized to all-NaN for C3PO's PCA projection.
    """
    global _context_ready
    if _context_ready:
        return
    t = np.asarray(analysis.t)
    if t.size < 2 or not np.isfinite(t).all() or np.any(np.diff(t) < 0):
        raise ValueError("Saved context timestamps must be finite and ordered.")
    context = np.asarray(analysis.c)
    valid = np.isfinite(context).all(axis=1)
    if valid.sum() < 2:
        raise ValueError("There are too few valid context samples to plot.")
    if np.any(~valid & ~np.isnan(context).all(axis=1)):
        analysis.c = context.copy()
        analysis.c[~valid] = np.nan
    analysis.interpolate_context(t=np.arange(t[0], t[-1], 0.001))
    valid_interp = np.isfinite(analysis.c_interp).all(axis=1)
    analysis.c_interp[~valid_interp] = np.nan
    if valid_interp.sum() < 2:
        raise ValueError("There are too few valid interpolated samples for PCA.")
    analysis.fit_context_pca(
        interpolated=True,
        n_components=min(analysis.context_dim, int(valid_interp.sum())),
    )
    _context_ready = True


def alignment_data(max_gap: float) -> tuple[np.ndarray, np.ndarray]:
    """Return cached PC scores with interpolation across long gaps masked.

    Parameters
    ----------
    max_gap : float
        Maximum permitted interpolation interval, in seconds.

    Returns
    -------
    scores : numpy.ndarray, shape (n_time, n_components)
        First three cached PC scores, with unsupported samples set to NaN.
    times : numpy.ndarray, shape (n_time,)
        Regular timestamps used by C3PO's aligned-response method.
    """
    t = np.asarray(analysis.t)
    times = analysis.t_interp
    right = np.searchsorted(t, times, side="left").clip(0, t.size - 1)
    left = np.maximum(right - 1, 0)
    unsupported = (t[right] != times) & ((t[right] - t[left]) > max_gap)
    scores = analysis.c_pca_interp[:, :3].copy()
    scores[unsupported] = np.nan
    return scores, times


def summarize_trials(
    event_column: str,
    group_column: str | None,
    before: float,
    after: float,
    numeric_bins: int = 0,
    max_gap: float = 0.1,
) -> dict[str, Any]:
    """Compute grouped trial-aligned means and SEM with bounded memory.

    Parameters
    ----------
    event_column : str
        Trial column containing event timestamps in seconds.
    group_column : str or None
        Scalar trial feature, or None to pool all trials.
    before, after : float
        Window lengths before and after the event, in seconds.
    numeric_bins : int, default=0
        Number of quantile bins for numeric grouping; zero uses exact values.
    max_gap : float, default=0.1
        Maximum interpolation gap in seconds.

    Returns
    -------
    dict of str to Any
        ``time`` has shape (n_time,). ``mean`` and ``sem`` have shape
        (n_groups, n_time, n_components). ``count`` has shape (n_groups, n_time)
        and reports the actual number of valid trials contributing at each time.
        ``trial_count`` counts complete valid windows in each group.
        ``available_count`` includes all trials with a nonmissing group label.
        C3PO excludes entire windows containing NaNs or exceeding recording
        bounds. Each group is processed separately in batches of 128.
    """
    prepare_context()
    event_times = pd.to_numeric(trials[event_column], errors="coerce").to_numpy(
        dtype=float, na_value=np.nan
    )
    if group_column is None:
        group_values = pd.Series("All trials", index=trials.index)
    else:
        group_values = trials[group_column]
        if numeric_bins:
            numeric = pd.to_numeric(group_values, errors="coerce")
            if numeric.notna().sum() != group_values.notna().sum():
                raise ValueError("Quantile bins require a numeric grouping feature.")
            if numeric.nunique() > 1:
                group_values = pd.qcut(numeric, q=numeric_bins, duplicates="drop")

    # Factorize nonmissing values without treating False or zero as missing.
    codes, labels = pd.factorize(group_values, sort=False)
    if not labels.size:
        raise ValueError("The selected grouping feature has no nonmissing values.")
    if labels.size > 12:
        raise ValueError(
            f"This feature has {labels.size} groups. Choose a categorical feature, "
            "or use numeric quantile bins under Plot settings."
        )
    scores, times = alignment_data(max_gap)
    dt = float(np.median(np.diff(times)))
    window = (int(-before / dt), int(after / dt))
    relative_time = np.arange(*window) * dt
    if not relative_time.size:
        raise ValueError("Choose a window at least one time bin wide.")
    shape = (labels.size, relative_time.size, scores.shape[1])
    total = np.zeros(shape)
    total_squared = np.zeros(shape)
    trial_count = np.zeros(labels.size, dtype=int)
    available_count = np.bincount(codes[codes >= 0], minlength=labels.size)
    valid = (
        np.isfinite(event_times)
        & (event_times >= times[0])
        & (event_times <= times[-1])
    )
    for group in range(labels.size):
        marks = event_times[valid & (codes == group)]
        for start in range(0, marks.size, 128):
            response = analysis.alligned_response(
                marks=marks[start : start + 128],
                t_plot=(-before, after),
                pca=True,
                passed_data=(scores, times),
            )
            if not response.size:
                continue
            total[group] += response.sum(axis=0)
            total_squared[group] += np.square(response).sum(axis=0)
            trial_count[group] += response.shape[0]
    count = np.broadcast_to(trial_count[:, None], shape[:2]).copy()
    n = count[..., None]
    mean = np.divide(total, n, out=np.full(shape, np.nan), where=n > 0)
    residual = np.maximum(total_squared - np.nan_to_num(mean) * total, 0.0)
    variance = np.divide(residual, n - 1, out=np.full(shape, np.nan), where=n > 1)
    sem = np.sqrt(np.divide(variance, n, out=np.full(shape, np.nan), where=n > 1))
    return dict(
        time=relative_time,
        mean=mean,
        sem=sem,
        count=count,
        trial_count=trial_count,
        available_count=available_count,
        labels=[str(label) for label in labels],
        excluded_trials=int(len(trials) - trial_count.sum()),
        align_to=event_column,
        group_by=group_column,
        numeric_bins=numeric_bins,
    )


def plot_context_overview() -> None:
    """Display a lightweight context preview when trial alignment is unavailable."""
    prepare_context()
    indices = np.linspace(
        0, len(analysis.t_interp) - 1, min(len(analysis.t_interp), 4000), dtype=int
    )
    projected = analysis.c_pca_interp[indices, :3]
    fig, axes = plt.subplots(
        projected.shape[1], 1, figsize=(10, 6), sharex=True, squeeze=False
    )
    for component, ax in enumerate(axes[:, 0]):
        ax.scatter(analysis.t_interp[indices], projected[:, component], s=2)
        ax.set_ylabel(f"Context PC {component + 1}")
        ax.spines[["top", "right"]].set_visible(False)
    axes[-1, 0].set_xlabel("Recording time (s)")
    fig.suptitle(f"{_analysis_name}: context overview")
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def update_plot(_: Any = None) -> None:
    """Render the selected trial-aligned preview and publish ``aligned_result``."""
    global aligned_result
    if analysis is None:
        return
    plot_button.disabled = True
    with plot_output:
        plot_output.clear_output(wait=True)
        try:
            if trials is None or trials.empty or align_to is None:
                plot_context_overview()
                return
            result = summarize_trials(
                align_to,
                group_by,
                before_widget.value,
                after_widget.value,
                bins_widget.value,
                gap_widget.value,
            )
            aligned_result = result
            fig, axes = plt.subplots(
                result["mean"].shape[2], 1, figsize=(10, 7), sharex=True, squeeze=False
            )
            palette = "tab10" if len(result["labels"]) <= 10 else "tab20"
            colors = plt.get_cmap(palette)(np.arange(len(result["labels"])))
            for component, ax in enumerate(axes[:, 0]):
                for group, label in enumerate(result["labels"]):
                    mean = result["mean"][group, :, component]
                    sem = result["sem"][group, :, component]
                    ax.plot(
                        result["time"],
                        mean,
                        color=colors[group],
                        label=f"{label} (n={result['trial_count'][group]})",
                    )
                    ax.fill_between(
                        result["time"],
                        mean - sem,
                        mean + sem,
                        color=colors[group],
                        alpha=0.15,
                    )
                ax.axvline(0, color="0.4", linewidth=1, linestyle="--")
                ax.set_ylabel(f"Context PC {component + 1}")
                ax.spines[["top", "right"]].set_visible(False)
            axes[0, 0].legend(
                title=group_by or "Trials", loc="upper left", bbox_to_anchor=(1, 1)
            )
            axes[-1, 0].set_xlabel(f"Time from {align_to} (s)")
            fig.suptitle(f"{_loaded_name}: trial-aligned context")
            fig.tight_layout()
            display(fig)
            plt.close(fig)
            print(
                "Mean ± SEM across trials; shaded intervals require at least two trials."
            )
            display(
                pd.DataFrame(
                    {
                        "Group": result["labels"],
                        "Total trials": result["available_count"],
                        "Trials retained": result["trial_count"],
                        "Minimum trials / time bin": result["count"].min(axis=1),
                        "Maximum trials / time bin": result["count"].max(axis=1),
                    }
                )
            )
            empty_groups = [
                label
                for label, count in zip(result["labels"], result["trial_count"])
                if count == 0
            ]
            if empty_groups:
                print(
                    "No complete valid windows for: "
                    + ", ".join(empty_groups)
                    + ". Try another event or a shorter window."
                )
            if result["excluded_trials"]:
                print(
                    f"Excluded {result['excluded_trials']:,} trials with missing values, out-of-range windows, or invalid context within the window."
                )
            plot_hint.value = ""
        except Exception as exc:
            aligned_result = None
            print(str(exc))
            with diagnostics:
                traceback.print_exc()
        finally:
            plot_button.disabled = align_to is None


def selection_changed(_: Any = None) -> None:
    """Synchronize trial selections with notebook variables and invalidate old plots."""
    global align_to, group_by, aligned_result
    align_to = align_widget.value
    group_by = group_widget.value
    aligned_result = None
    if not _configuring:
        plot_output.clear_output()
        plot_hint.value = "<i>Selections changed. Click <b>Update plot</b>.</i>"


def configure_trials() -> None:
    """Populate trial widgets, using start_time and reward when available."""
    global _configuring
    _configuring = True
    try:
        columns = (
            scalar_columns(trials) if trials is not None and not trials.empty else []
        )
        numeric_columns = [
            name
            for name in columns
            if not pd.api.types.is_bool_dtype(trials[name])
            and pd.to_numeric(trials[name], errors="coerce").notna().any()
        ]
        numeric_columns.sort(
            key=lambda name: (name != "start_time", "time" not in name.lower(), name)
        )
        align_widget.options = [(name, name) for name in numeric_columns] or [
            ("No event-time columns", None)
        ]
        align_widget.value = numeric_columns[0] if numeric_columns else None
        group_widget.options = [("All trials", None)] + [
            (name, name) for name in columns
        ]
        group_widget.value = "reward" if "reward" in columns else None
        bins_widget.value = 0
        align_widget.disabled = not numeric_columns
        group_widget.disabled = not numeric_columns
        plot_button.disabled = not numeric_columns
        trial_note.value = (
            f"<b>{len(trials):,} trials.</b> Choose an event-time column in seconds and a trial feature."
            if numeric_columns
            else "<b>No usable trials table.</b> Showing a context overview; the loaded model and NWB remain available."
        )
        with trial_preview:
            trial_preview.clear_output()
            if trials is not None:
                print("Trial columns:", ", ".join(trials.columns))
                display(trials.head(6))
        selection_changed()
    finally:
        _configuring = False


def load_recording(_: Any = None) -> None:
    """Load the selected model and NWB file, then open the trial explorer."""
    global analysis, nwb_io, nwbfile, trials, selected_model, _context_ready
    global _analysis_name, _loaded_name, _busy, mark_times, mark_ids, aligned_result
    global align_to, group_by
    if model_widget.value is None or _busy:
        return
    if _loaded_name == model_widget.value and nwb_io is not None:
        status.value = "<b>This recording is already loaded.</b> Explore it below."
        return
    _busy = True
    load_button.disabled = model_widget.disabled = True
    explorer.layout.display = "none"
    spike_button.disabled = True
    progress.layout.display = ""
    progress.value = 0
    progress.bar_style = "info"
    diagnostics.clear_output()
    plot_output.clear_output()
    spike_output.clear_output()
    try:
        if nwb_io is not None:
            nwb_io.close()
        nwb_io = nwbfile = trials = None
        align_to = group_by = None
        _loaded_name = None
        mark_times = mark_ids = aligned_result = None
        selected_model = (
            available_models.loc[available_models["model_name"].eq(model_widget.value)]
            .iloc[0]
            .to_dict()
        )
        status.value = "<b>1/3 · Loading pretrained model…</b>"
        if _analysis_name != selected_model["model_name"] or analysis is None:
            analysis = None
            _context_ready = False
            _analysis_name = None
            gc.collect()
            with diagnostics:
                analysis = C3poAnalysis.from_cloud_model(
                    repo_name=HF_REPO,
                    model_name=selected_model["model_name"],
                    cache_dir=HF_CACHE_DIR,
                    revision=HF_REVISION,
                    token=False,
                )
            _analysis_name = selected_model["model_name"]
        progress.value = 1
        status.value = "<b>2/3 · Opening the NWB recording and reading trials…</b>"
        with diagnostics:
            nwb_io, nwbfile = get_dandi_file(
                dandiset_id=selected_model["dandiset_id"],
                dandi_path=selected_model["dandi_path"],
                dandi_instance=selected_model["dandi_instance"],
            )
            if "trials" in nwbfile.intervals:
                trials = nwbfile.intervals["trials"].to_dataframe()
            elif nwbfile.trials is not None:
                trials = nwbfile.trials.to_dataframe()
        progress.value = 2
        status.value = "<b>3/3 · Preparing the context preview…</b>"
        prepare_context()
        _loaded_name = selected_model["model_name"]
        configure_trials()
        explorer.layout.display = ""
        spike_button.disabled = False
        update_plot()
        progress.value = 3
        progress.bar_style = "success"
        status.value = (
            f"<b>Loaded {html.escape(_loaded_name)}</b> · "
            f"{len(analysis.t):,} context samples · "
            f"{len(trials) if trials is not None else 0:,} trials"
        )
    except Exception as exc:
        progress.bar_style = "danger"
        status.value = f"<b>Could not finish loading:</b> {html.escape(str(exc))}. You can retry with Load recording."
        with diagnostics:
            traceback.print_exc()
    finally:
        _busy = False
        load_button.disabled = model_widget.disabled = False


def load_spikes(_: Any = None) -> None:
    """Optionally load raw spike arrays; reuse them on subsequent clicks."""
    global mark_times, mark_ids
    if nwbfile is None:
        return
    spike_button.disabled = True
    with spike_output:
        spike_output.clear_output(wait=True)
        try:
            if mark_times is None:
                mark_times, mark_ids = get_spike_data(nwbfile)
                mark_ids = mark_ids.astype(np.int64, copy=False)
            print(f"Loaded {mark_times.size:,} spikes into mark_times and mark_ids.")
        except Exception as exc:
            print(str(exc))
        finally:
            spike_button.disabled = False


def model_changed(_: Any = None) -> None:
    """Show the selected recording without downloading anything."""
    load_button.disabled = model_widget.value is None
    if model_widget.value is None:
        model_details.value = "Choose a recording to begin."
    else:
        row = available_models.loc[
            available_models["model_name"].eq(model_widget.value)
        ].iloc[0]
        model_details.value = f"<small>Dandiset {html.escape(row.dandiset_id)} · {html.escape(row.dandi_path)}</small>"


print("Loading the model list…")
manifest_path = hf_hub_download(
    repo_id=HF_REPO,
    repo_type="dataset",
    filename="dandi_model_manifest.csv",
    revision=HF_REVISION,
    cache_dir=str(HF_CACHE_DIR),
    token=False,
)
manifest = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
required = ["model_name", "dandiset_id", "dandi_path", "dandi_instance"]
if not set(required).issubset(manifest.columns) or manifest.empty:
    raise ValueError("The model manifest is empty or missing required columns.")
manifest[required] = manifest[required].apply(lambda column: column.str.strip())
if manifest[required].eq("").any().any() or manifest.model_name.duplicated().any():
    raise ValueError(
        "The model manifest contains blank fields or duplicate model names."
    )
if not manifest.dandiset_id.str.fullmatch(r"\d{6}").all():
    raise ValueError("Expected six-digit Dandiset IDs.")
repo_files = set(
    HfApi().list_repo_files(
        repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION, token=False
    )
)
complete = [
    all(
        f"{name}/{filename}" in repo_files
        for filename in ("args.json", "params.hdf5", "embedding.npz")
    )
    for name in manifest.model_name
]
available_models = (
    manifest.loc[complete].sort_values("model_name").reset_index(drop=True)
)
if available_models.empty:
    raise ValueError("No complete pretrained models are currently available.")

style = {"description_width": "initial"}
model_widget = widgets.Dropdown(
    options=[("Choose a recording…", None)]
    + [(row.model_name, row.model_name) for row in available_models.itertuples()],
    value=None,
    description="Recording",
    style=style,
    layout=widgets.Layout(width="70%"),
)
load_button = widgets.Button(
    description="Load recording", button_style="primary", icon="download", disabled=True
)
model_details = widgets.HTML("Choose a recording to begin.")
status = widgets.HTML(
    f"<b>{len(available_models)} recordings available.</b> Choose one, then click Load recording."
)
progress = widgets.IntProgress(
    min=0, max=3, layout=widgets.Layout(display="none", width="95%")
)
align_widget = widgets.Dropdown(
    options=[("Load a recording first", None)],
    description="Align to",
    style=style,
    disabled=True,
)
group_widget = widgets.Dropdown(
    options=[("All trials", None)], description="Group by", style=style, disabled=True
)
plot_button = widgets.Button(
    description="Update plot", button_style="primary", icon="line-chart", disabled=True
)
trial_note = widgets.HTML()
plot_hint = widgets.HTML()
before_widget = widgets.FloatSlider(
    value=0.5,
    min=0,
    max=5,
    step=0.1,
    description="Seconds before",
    style=style,
    continuous_update=False,
)
after_widget = widgets.FloatSlider(
    value=1,
    min=0.1,
    max=10,
    step=0.1,
    description="Seconds after",
    style=style,
    continuous_update=False,
)
bins_widget = widgets.Dropdown(
    options=[("Exact values", 0), ("4 quantile bins", 4), ("6 quantile bins", 6)],
    description="Numeric groups",
    style=style,
)
gap_widget = widgets.FloatSlider(
    value=0.1,
    min=0.02,
    max=1,
    step=0.02,
    description="Max gap (s)",
    style=style,
    continuous_update=False,
)
plot_output = widgets.Output()
trial_preview = widgets.Output()
diagnostics = widgets.Output()
spike_output = widgets.Output()
spike_button = widgets.Button(
    description="Load raw spikes", icon="download", disabled=True
)
plot_settings = widgets.Accordion(
    children=[widgets.VBox([before_widget, after_widget, bins_widget, gap_widget])],
    selected_index=None,
)
plot_settings.set_title(0, "Plot settings")
trial_details = widgets.Accordion(children=[trial_preview], selected_index=None)
trial_details.set_title(0, "Preview trial columns and rows")
explorer = widgets.VBox(
    [
        widgets.HTML("<h3>Explore trial-aligned context</h3>"),
        trial_note,
        widgets.HBox(
            [align_widget, group_widget], layout=widgets.Layout(flex_flow="row wrap")
        ),
        plot_settings,
        widgets.HBox([plot_button, plot_hint]),
        plot_output,
        trial_details,
    ],
    layout=widgets.Layout(display="none"),
)
notes = widgets.HTML(
    "<p>C3PO interpolates context onto a 20 ms grid and fits PCA once per model. "
    "The first three PCs share the same basis across all groups and alignments. "
    "C3PO's alligned_response excludes entire trials with incomplete or invalid "
    "windows, including gaps exceeding Max gap. Counts are shown for every group. "
    "Event times "
    "must use the same seconds reference as the saved context.</p>"
    "<p><code>analysis</code>, <code>nwbfile</code>, <code>trials</code>, "
    "<code>align_to</code>, <code>group_by</code>, and <code>aligned_result</code> "
    "are available for further analysis. The NWB handle remains open. "
    "Raw spike extraction is optional and may read substantial data.</p>"
)
with diagnostics:
    unavailable = manifest.loc[~np.asarray(complete), "model_name"].tolist()
    if unavailable:
        print("Incomplete model entries omitted:", ", ".join(unavailable))
extras = widgets.Accordion(
    children=[widgets.VBox([notes, spike_button, spike_output]), diagnostics],
    selected_index=None,
)
extras.set_title(0, "Details and optional raw spikes")
extras.set_title(1, "Loading log / troubleshooting")

for control in [
    align_widget,
    group_widget,
    before_widget,
    after_widget,
    bins_widget,
    gap_widget,
]:
    control.observe(selection_changed, names="value")
model_widget.observe(model_changed, names="value")
load_button.on_click(load_recording)
plot_button.on_click(update_plot)
spike_button.on_click(load_spikes)
_explorer_ui = widgets.VBox(
    [
        widgets.HBox(
            [model_widget, load_button], layout=widgets.Layout(flex_flow="row wrap")
        ),
        model_details,
        status,
        progress,
        explorer,
        extras,
    ]
)
_explorer_widgets = [
    value for value in list(globals().values()) if isinstance(value, widgets.Widget)
]
display(_explorer_ui)

Models and recording paths come from the [live manifest](https://huggingface.co/datasets/samuelbray32/c3po-publication/blob/main/dandi_model_manifest.csv). Model loading uses `C3poAnalysis.from_cloud_model`; NWB access uses `c3po.utils.dandi.get_dandi_file`. Downloads are cached.

For your own analysis, add a code cell and use `analysis`, `nwbfile`, `trials`, `align_to`, and `group_by`. `aligned_result` contains the preview time grid, group means, SEM, and sample counts after plotting. PCA is fitted once with `analysis.fit_context_pca(interpolated=True)` after `analysis.interpolate_context` creates a 20 ms grid. Plots overlay every group using `analysis.alligned_response`; the PCA and projections remain available on `analysis`. When finished reading NWB data, run `nwb_io.close()`.